In [1]:
import pandas as pd
from utils import compute_pass_metrics, compute_summary_stats, load_all_results

all_results = load_all_results()
print(f"Loaded results for {len(all_results)} setups")

Loaded results for 7 setups


In [2]:
# Select a specific model for detailed multi-run analysis
ANALYSIS_MODEL = "haiku-4-5"

analysis_model_df = all_results[ANALYSIS_MODEL]
model_stats = compute_summary_stats(analysis_model_df)
pass_metrics = compute_pass_metrics(analysis_model_df)

print(f"Multi-Run Analysis: {ANALYSIS_MODEL}")
print(f"{'=' * 60}")
print(f"Runs: {model_stats['n_runs']} | Instances: {model_stats['n_instances']}")

Multi-Run Analysis: haiku-4-5
Runs: 11 | Instances: 55


## 1. Mean vs Consistent Resolution

**Are we measuring the wrong thing?**

The mean resolution rate answers: *"On average, what's the expected success rate?"*

But there's another question: *"Which instances can the agent **reliably** solve?"*

| Metric | Definition | What it captures |
|--------|------------|------------------|
| **Mean** | Average resolution rate across runs | Expected single-attempt performance |
| **pass@k** | 1 - C(n-c,k)/C(n,k) | Probability of ≥1 success when sampling k |
| **pass^k** | C(c,k)/C(n,k) | Probability that k random samples ALL succeed |

**Why this matters for BC-Bench:**
- If an instance is resolved in 1 of 10 runs, the mean counts it as 10% solved
- But is that a "real" capability or just luck (temperature sampling, etc.)?
- pass^k is more conservative but arguably more meaningful for claims like "the agent can solve X"

In [3]:
# Compute pass metrics for our analysis model (using utils)
metrics = compute_pass_metrics(analysis_model_df)

print(f"Resolution Metrics Comparison: {ANALYSIS_MODEL}")
print(f"{'=' * 65}")
print(f"Runs: {metrics['n_runs']} | Instances: {metrics['n_instances']}")
print()
print(f"{'Metric':<30} {'Percentage':>15}")
print(f"{'-' * 65}")
print(f"{'Mean':<30} {metrics['mean_pct']:>14.2f}%")
print(f"{'pass@k (1-C(n-c,k)/C(n,k))':<30} {metrics['pass_at_k'] * 100:>14.2f}%")
print(f"{'pass^k (C(c,k)/C(n,k))':<30} {metrics['pass_hat_k'] * 100:>14.2f}%")
print(f"{'-' * 65}")
print()
print(f"📊 Reliability gap (pass@k - pass^k): {(metrics['pass_at_k'] - metrics['pass_hat_k']) * 100:.1f}%")
print("   This shows the spread between optimistic and pessimistic estimates")

Resolution Metrics Comparison: haiku-4-5
Runs: 11 | Instances: 55

Metric                              Percentage
-----------------------------------------------------------------
Mean                                    45.12%
pass@k (1-C(n-c,k)/C(n,k))              76.36%
pass^k (C(c,k)/C(n,k))                  21.82%
-----------------------------------------------------------------

📊 Reliability gap (pass@k - pass^k): 54.5%
   This shows the spread between optimistic and pessimistic estimates


In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from bcbench.results.metrics import pass_at_k, pass_hat_k

# Visualize: pass^k vs pass@k comparison using correct formulas
run_ids_sorted = sorted(analysis_model_df["run_id"].unique())
pivot = analysis_model_df.pivot_table(index="instance_id", columns="run_id", values="resolved", aggfunc=lambda x: x.iloc[0])
pivot = pivot[run_ids_sorted]

n_runs = len(run_ids_sorted)
n_instances = len(pivot)

fig = make_subplots(rows=1, cols=2, subplot_titles=("pass^k: C(c,k)/C(n,k)", "pass@k: 1-C(n-c,k)/C(n,k)"))

k_values = list(range(1, n_runs + 1))

# Left: pass^k - measures the probability that all k trials succeed
pass_hat_k_pcts = []
for k in k_values:
    first_k_runs = run_ids_sorted[:k]
    pivot_k = pivot[first_k_runs]
    total = sum(pass_hat_k(k, int(row.sum()), k) for _, row in pivot_k.iterrows())
    pass_hat_k_pcts.append(total / n_instances * 100)

fig.add_trace(go.Bar(x=k_values, y=pass_hat_k_pcts, name="pass^k", marker_color="coral"), row=1, col=1)

# Right: pass@k - probability of at least 1 success
pass_at_k_pcts = []
for k in k_values:
    first_k_runs = run_ids_sorted[:k]
    pivot_k = pivot[first_k_runs]
    total = sum(pass_at_k(k, int(row.sum()), k) for _, row in pivot_k.iterrows())
    pass_at_k_pcts.append(total / n_instances * 100)

fig.add_trace(go.Bar(x=k_values, y=pass_at_k_pcts, name="pass@k", marker_color="steelblue"), row=1, col=2)

# Add mean line to both for reference
mean_pct = metrics["mean_pct"]
fig.add_hline(y=mean_pct, line_dash="dash", line_color="black", row=1, col=1, annotation_text=f"Mean: {mean_pct:.1f}%", annotation_position="top right")  # type: ignore
fig.add_hline(y=mean_pct, line_dash="dash", line_color="black", row=1, col=2)  # type: ignore

fig.update_xaxes(title_text="k (number of runs)", row=1, col=1)
fig.update_yaxes(title_text="Probability (%)", row=1, col=1)
fig.update_xaxes(title_text="k (number of runs)", row=1, col=2)
fig.update_yaxes(title_text="Probability (%)", row=1, col=2)

# Same y-axis range for fair comparison
fig.update_yaxes(range=[0, 100])

fig.update_layout(height=400, title_text=f"pass^k vs pass@k: {ANALYSIS_MODEL} ({n_runs} runs)", showlegend=False)
fig.show()

print("\n🔍 Interpretation:")
print("   • pass^k (left): Probability that k random samples would ALL succeed")
print(f"     - k=1: {pass_hat_k_pcts[0]:.1f}% (same as mean)")
print(f"     - k={n_runs}: {pass_hat_k_pcts[-1]:.1f}% (all runs succeed)")
print("   • pass@k (right): Probability of at least 1 success in k samples")
print(f"     - k=1: {pass_at_k_pcts[0]:.1f}% (same as mean)")
print(f"     - k={n_runs}: {pass_at_k_pcts[-1]:.1f}% (any success)")
print(f"\n   📊 Reliability gap at k={n_runs}: {pass_at_k_pcts[-1] - pass_hat_k_pcts[-1]:.1f}%")


🔍 Interpretation:
   • pass^k (left): Probability that k random samples would ALL succeed
     - k=1: 41.8% (same as mean)
     - k=11: 21.8% (all runs succeed)
   • pass@k (right): Probability of at least 1 success in k samples
     - k=1: 41.8% (same as mean)
     - k=11: 76.4% (any success)

   📊 Reliability gap at k=11: 54.5%


## 2. Normality Testing

**Why it matters:** The t-distribution CI (SEM based calculation) assumes the per-run means are normally distributed. If they're not, our confidence intervals may be wrong.

**The catch:** With only ~11 runs, normality tests have LOW POWER - they often can't detect non-normality even when it exists. So "passing" Shapiro-Wilk doesn't prove normality, it just means we couldn't prove it's NOT normal.

In [5]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import probplot, shapiro

# Get per-run resolution rates (this is what we're testing for normality)
per_run_resolved = analysis_model_df.groupby("run_id")["resolved"].mean() * 100
n_runs = len(per_run_resolved)

print(f"Sample size: {n_runs} runs")
print(f"Per-run resolution rates: {np.array(per_run_resolved.values).round(1)}")
print(f"Mean: {per_run_resolved.mean():.2f}%, SD: {per_run_resolved.std(ddof=1):.2f}%")
print()

# Shapiro-Wilk test (best for small samples)
stat, p_value = shapiro(per_run_resolved)
print(f"Shapiro-Wilk test: W={stat:.4f}, p-value={p_value:.4f}")
if p_value > 0.05:
    print("→ p > 0.05: Cannot reject normality (but doesn't prove it!)")
else:
    print("→ p ≤ 0.05: Evidence against normality")

# Visual: Q-Q plot and histogram
fig = make_subplots(rows=1, cols=2, subplot_titles=("Q-Q Plot (Normal)", "Distribution of Per-Run Means"))

# Q-Q plot
probplot_result = probplot(per_run_resolved, dist="norm")
theoretical_quantiles = probplot_result[0]
slope, intercept = float(probplot_result[1][0]), float(probplot_result[1][1])  # type: ignore[arg-type]

fig.add_trace(go.Scatter(x=theoretical_quantiles[0], y=theoretical_quantiles[1], mode="markers", name="Data", marker={"size": 10}), row=1, col=1)
# Add reference line
x_line = np.array([theoretical_quantiles[0].min(), theoretical_quantiles[0].max()])
y_line = intercept + slope * x_line
fig.add_trace(go.Scatter(x=x_line, y=y_line, mode="lines", name="Normal ref", line={"dash": "dash", "color": "red"}), row=1, col=1)

# Histogram
fig.add_trace(go.Histogram(x=per_run_resolved, nbinsx=6, name="Observed", showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Theoretical Quantiles", row=1, col=1)
fig.update_yaxes(title_text="Sample Quantiles", row=1, col=1)
fig.update_xaxes(title_text="Resolution Rate (%)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_layout(height=400, title_text=f"Normality Check for Per-Run Means (n={n_runs})", showlegend=False)
fig.show()

print("\n⚠️  IMPORTANT: With n=11, Shapiro-Wilk has ~50% power to detect moderate non-normality.")
print("   Visual inspection of Q-Q plot is often more informative than the test.")

Sample size: 11 runs
Per-run resolution rates: [41.8 45.5 40.  52.7 43.6 43.6 49.1 47.3 47.3 43.6 41.8]
Mean: 45.12%, SD: 3.71%

Shapiro-Wilk test: W=0.9474, p-value=0.6106
→ p > 0.05: Cannot reject normality (but doesn't prove it!)



⚠️  IMPORTANT: With n=11, Shapiro-Wilk has ~50% power to detect moderate non-normality.
   Visual inspection of Q-Q plot is often more informative than the test.


## 3. SEM-based CI vs Bootstrap CI

**SEM approach (what we're using):**
- Assumes per-run means are normally distributed
- CI = mean ± t × (SD / √n)
- Works well if normality holds

**Bootstrap approach:**
- No normality assumption needed
- Resample with replacement many times, compute mean each time
- CI from percentiles of resampled means
- More robust for small/non-normal samples

**Which to use?** Bootstrap is generally safer for small samples. If both give similar CIs, you can trust either.

In [6]:
from typing import Any

from scipy import stats as scipy_stats


def bootstrap_ci(data: np.ndarray, n_bootstrap: int = 10000, ci_level: float = 0.95) -> dict[str, Any]:
    """Compute bootstrap confidence interval using percentile method."""
    rng = np.random.default_rng(42)  # reproducible
    bootstrap_means = np.array([rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_bootstrap)])
    alpha = 1 - ci_level
    ci_low = np.percentile(bootstrap_means, alpha / 2 * 100)
    ci_high = np.percentile(bootstrap_means, (1 - alpha / 2) * 100)
    return {
        "mean": data.mean(),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "ci_half": (ci_high - ci_low) / 2,
        "bootstrap_means": bootstrap_means,
    }


def sem_ci(data: np.ndarray, ci_level: float = 0.95) -> dict[str, float]:
    """Compute t-distribution confidence interval."""
    n = len(data)
    mean = float(data.mean())
    sem = float(data.std(ddof=1)) / np.sqrt(n)
    t_crit = scipy_stats.t.ppf((1 + ci_level) / 2, df=n - 1)
    ci_half = t_crit * sem
    return {
        "mean": mean,
        "ci_low": mean - ci_half,
        "ci_high": mean + ci_half,
        "ci_half": ci_half,
    }


# Compare both methods
data = np.array(per_run_resolved.values)
sem_result = sem_ci(data)
boot_result = bootstrap_ci(data)

print("Comparison: SEM-based vs Bootstrap CI (95%)")
print("=" * 60)
print(f"{'Method':<15} {'Mean':>10} {'CI Low':>10} {'CI High':>10} {'CI ±':>10}")
print("-" * 60)
print(f"{'SEM (t-dist)':<15} {sem_result['mean']:>10.2f} {sem_result['ci_low']:>10.2f} {sem_result['ci_high']:>10.2f} {sem_result['ci_half']:>10.2f}")
print(f"{'Bootstrap':<15} {boot_result['mean']:>10.2f} {boot_result['ci_low']:>10.2f} {boot_result['ci_high']:>10.2f} {boot_result['ci_half']:>10.2f}")
print("-" * 60)

diff = abs(sem_result["ci_half"] - boot_result["ci_half"])
print(f"\nDifference in CI width: {diff:.2f}%")
if diff < 1:
    print("→ Methods agree well - either is reasonable to use")
else:
    print("→ Notable difference - bootstrap may be more reliable")

# Visualize bootstrap distribution
fig = go.Figure()
fig.add_trace(go.Histogram(x=boot_result["bootstrap_means"], nbinsx=50, name="Bootstrap means"))
fig.add_vline(x=sem_result["ci_low"], line_dash="dash", line_color="red", annotation_text="SEM CI low")
fig.add_vline(x=sem_result["ci_high"], line_dash="dash", line_color="red", annotation_text="SEM CI high")
fig.add_vline(x=boot_result["ci_low"], line_dash="dot", line_color="blue", annotation_text="Boot CI low")
fig.add_vline(x=boot_result["ci_high"], line_dash="dot", line_color="blue", annotation_text="Boot CI high")
fig.update_layout(
    title="Bootstrap Distribution of Mean Resolution Rate",
    xaxis_title="Mean Resolution Rate (%)",
    yaxis_title="Count",
    height=400,
)
fig.show()

Comparison: SEM-based vs Bootstrap CI (95%)
Method                Mean     CI Low    CI High       CI ±
------------------------------------------------------------
SEM (t-dist)         45.12      42.63      47.62       2.49
Bootstrap            45.12      43.14      47.27       2.07
------------------------------------------------------------

Difference in CI width: 0.43%
→ Methods agree well - either is reasonable to use


## 4. Bootstrap with Different Sample Sizes

**What happens with fewer runs?**

- **1 run:** Bootstrap is MEANINGLESS. You're resampling one number → always get that same number. CI width = 0 (falsely confident!)
- **2 runs:** Very unstable. Only 2 unique values to resample from.
- **3 runs:** Technically works but very wide CIs and unreliable.
- **4+ runs:** Starts to become reasonable.

This shows why single-run benchmarks are dangerous for claims like "Model A is X% better than Model B".

In [7]:
run_ids = sorted(analysis_model_df["run_id"].unique())
full_data = np.array(per_run_resolved.values)

print("Bootstrap CI by Number of Runs (using first N runs)")
print("=" * 55)
print(f"{'Runs':<6} {'Mean':>8} {'Boot ±':>10} {'SEM ±':>10} {'Comment':<20}")
print("-" * 55)

for n in [1, 2, 3, 4, 5, 6, 8, len(run_ids)]:
    if n > len(run_ids):
        continue
    subset_data = full_data[:n]

    if n == 1:
        print(f"{n:<6} {subset_data[0]:>8.1f} {'N/A':>10} {'N/A':>10} {'Only 1 point!':<20}")
        continue

    boot = bootstrap_ci(subset_data)
    sem = sem_ci(subset_data)

    comment = ""
    if n == 2:
        comment = "Very unstable"
    elif n == 3:
        comment = "Still unreliable"
    elif n <= 5:
        comment = "Borderline"

    print(f"{n:<6} {boot['mean']:>8.1f} {boot['ci_half']:>10.1f} {sem['ci_half']:>10.1f} {comment:<20}")

print("-" * 55)
print("\n⚠️  Notice how CI width shrinks as n increases (more confident).")
print("   With 1-3 runs, you can't make reliable statistical claims.")

Bootstrap CI by Number of Runs (using first N runs)
Runs       Mean     Boot ±      SEM ± Comment             
-------------------------------------------------------
1          41.8        N/A        N/A Only 1 point!       
2          43.6        1.8       23.1 Very unstable       
3          42.4        2.7        6.9 Still unreliable    
4          45.0        4.5        9.0 Borderline          
5          44.7        3.8        6.1 Borderline          
6          44.5        3.2        4.6                     
8          45.5        2.6        3.4                     
11         45.1        2.1        2.5                     
-------------------------------------------------------

⚠️  Notice how CI width shrinks as n increases (more confident).
   With 1-3 runs, you can't make reliable statistical claims.


## 5. Bootstrap Runs vs Bootstrap Instances

**Two different questions:**

| Approach | What you resample | What it measures |
|----------|-------------------|------------------|
| **Bootstrap Runs** | The per-run means (11 numbers) | Run-to-run variance (randomness in agent behavior) |
| **Bootstrap Instances** | Individual instance results within run(s) | Instance difficulty variance + some run variance |

**Why does it matter?**
- **Bootstrap runs**: Captures the "luck" factor - same agent on same task might get different results due to sampling, temperature, etc.
- **Bootstrap instances**: Captures "what if we had different test cases?" - useful if you want to generalize to unseen problems.

**The key insight**: With only 1 run, you CAN bootstrap instances, but you're measuring the WRONG variance. You're measuring "how spread out are my instance difficulties" not "how reliable is my agent's performance".

In [8]:
def bootstrap_instances(df: pd.DataFrame, n_bootstrap: int = 10000, ci_level: float = 0.95) -> dict[str, Any]:
    """Bootstrap by resampling instances (rows) within the dataframe."""
    rng = np.random.default_rng(42)
    n_instances = len(df)
    resolved_values = np.array(df["resolved"].values)

    bootstrap_means = np.array([rng.choice(resolved_values, size=n_instances, replace=True).mean() * 100 for _ in range(n_bootstrap)])

    alpha = 1 - ci_level
    ci_low = np.percentile(bootstrap_means, alpha / 2 * 100)
    ci_high = np.percentile(bootstrap_means, (1 - alpha / 2) * 100)
    return {
        "mean": float(resolved_values.mean()) * 100,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "ci_half": (ci_high - ci_low) / 2,
        "n_instances": n_instances,
    }


# Compare bootstrap instances vs bootstrap runs for different numbers of runs
run_ids_sorted = sorted(analysis_model_df["run_id"].unique())
n_instances_per_run = analysis_model_df.groupby("run_id").size().iloc[0]

print("Comparison: Bootstrap Instances vs Bootstrap Runs")
print("=" * 85)
print(f"Each run has {n_instances_per_run} instances")
print()
print(f"{'Runs':<6} {'Instances':<10} {'Mean':>8} {'Boot Inst ±':>12} {'Boot Runs ±':>12} {'Difference':>12}")
print("-" * 85)

for n in [1, 2, 3, 5, len(run_ids_sorted)]:
    if n > len(run_ids_sorted):
        continue

    # Get data for first N runs
    subset_run_ids = run_ids_sorted[:n]
    subset_df = analysis_model_df[analysis_model_df["run_id"].isin(subset_run_ids)]

    # Bootstrap instances
    boot_inst = bootstrap_instances(subset_df)

    # Bootstrap runs (need at least 2 runs)
    if n >= 2:
        per_run_means = np.array(subset_df.groupby("run_id")["resolved"].mean().values) * 100
        boot_runs = bootstrap_ci(per_run_means)
        boot_runs_ci = f"{boot_runs['ci_half']:.2f}"
        diff = abs(boot_inst["ci_half"] - boot_runs["ci_half"])
        diff_str = f"{diff:.2f}"
    else:
        boot_runs_ci = "N/A"
        diff_str = "N/A"

    print(f"{n:<6} {boot_inst['n_instances']:<10} {boot_inst['mean']:>8.2f} {boot_inst['ci_half']:>12.2f} {boot_runs_ci:>12} {diff_str:>12}")

print("-" * 85)
print()
print("🔍 Key observations:")
print("   • Bootstrap Instances gives NARROW CIs even with 1 run (misleading!)")
print("   • Bootstrap Runs gives WIDE CIs that shrink as runs increase (correct behavior)")
print("   • The difference shows how much you'd UNDERESTIMATE uncertainty with 1 run")

Comparison: Bootstrap Instances vs Bootstrap Runs
Each run has 55 instances

Runs   Instances      Mean  Boot Inst ±  Boot Runs ±   Difference
-------------------------------------------------------------------------------------
1      55            41.82        12.73          N/A          N/A
2      110           43.64         9.09         1.82         7.27
3      165           42.42         7.58         2.73         4.85
5      275           44.73         5.82         3.82         2.00
11     605           45.12         3.97         2.07         1.90
-------------------------------------------------------------------------------------

🔍 Key observations:
   • Bootstrap Instances gives NARROW CIs even with 1 run (misleading!)
   • Bootstrap Runs gives WIDE CIs that shrink as runs increase (correct behavior)
   • The difference shows how much you'd UNDERESTIMATE uncertainty with 1 run


In [9]:
# Visualize the difference
fig = make_subplots(rows=1, cols=2, subplot_titles=("1 Run: Bootstrap Instances (WRONG)", f"{len(run_ids_sorted)} Runs: Bootstrap Runs (CORRECT)"))

# 1 run - bootstrap instances (misleading narrow CI)
one_run_df = analysis_model_df[analysis_model_df["run_id"] == run_ids_sorted[0]]
rng = np.random.default_rng(42)
one_run_resolved = np.array(one_run_df["resolved"].values)
boot_means_1run = [rng.choice(one_run_resolved, size=len(one_run_df), replace=True).mean() * 100 for _ in range(10000)]

fig.add_trace(go.Histogram(x=boot_means_1run, nbinsx=50, name="1 run (instances)", marker_color="red"), row=1, col=1)

# All runs - bootstrap runs (correct)
fig.add_trace(go.Histogram(x=boot_result["bootstrap_means"], nbinsx=50, name=f"{len(run_ids_sorted)} runs", marker_color="blue"), row=1, col=2)

# Add actual mean line to both
actual_mean = analysis_model_df["resolved"].mean() * 100
fig.add_vline(x=actual_mean, line_dash="dash", line_color="black")

fig.update_xaxes(title_text="Mean Resolution Rate (%)", row=1, col=1)
fig.update_xaxes(title_text="Mean Resolution Rate (%)", row=1, col=2)
fig.update_layout(height=400, title_text="Why Single-Run Bootstrap is Misleading", showlegend=False)

# Make x-axes same range for fair comparison
x_min = min(*boot_means_1run, *boot_result["bootstrap_means"]) - 2
x_max = max(*boot_means_1run, *boot_result["bootstrap_means"]) + 2
fig.update_xaxes(range=[x_min, x_max])

fig.show()

print("\n⚠️  The left plot looks more 'precise' but it's FALSE PRECISION!")
print("   It only captures instance variance, not run-to-run variance.")
print("   The right plot shows the TRUE uncertainty in your estimate.")


⚠️  The left plot looks more 'precise' but it's FALSE PRECISION!
   It only captures instance variance, not run-to-run variance.
   The right plot shows the TRUE uncertainty in your estimate.


### Why not always use Bootstrap?

**You absolutely can!** Your intuition is correct. Here's the full picture:

| Factor | SEM (t-distribution) | Bootstrap |
|--------|---------------------|-----------|
| Assumptions | Normality required | None (non-parametric) |
| Computation | Instant (closed-form) | Slower (10k+ resamples) |
| Interpretability | Familiar formula | "Black box" to some |
| Small samples | Can be off if non-normal | More reliable |
| Extreme data | Sensitive to outliers | Also sensitive, but can use trimmed mean |

**Practical recommendation:**
- **For exploration/quick checks**: SEM is fine
- **For publication/claims**: Use bootstrap (or report both)
- **When they disagree**: Trust bootstrap

**The real reason people still use SEM**: Legacy. The t-test was invented in 1908 (by a Guinness brewery statistician!). Bootstrap came in 1979. Old habits die hard.

**Bottom line**: Just use bootstrap. You're not missing anything by skipping SEM.

## 6. Benchmark Results

In [10]:
# Summary with pass metrics (using first K runs for fair comparison)
K_RUNS = 5

summary_tok_data = []
for model_name, model_df in sorted(all_results.items()):
    n_runs_available = model_df["run_id"].nunique()
    if n_runs_available < K_RUNS:
        continue

    m = compute_pass_metrics(model_df, k=K_RUNS)
    agent = model_df["agent_name"].iloc[0] if "agent_name" in model_df.columns else "Unknown"

    summary_tok_data.append(
        {
            "Model": model_name,
            "Agent": agent,
            "Mean %": f"{m['mean_pct']:.1f}",
            f"pass@{K_RUNS} %": f"{m['pass_at_k'] * 100:.1f}",
            f"pass^{K_RUNS} %": f"{m['pass_hat_k'] * 100:.1f}",
        }
    )

summary_tok_df = pd.DataFrame(summary_tok_data)
print(f"Pass Metrics Summary (first {K_RUNS} runs per model)")
print("=" * 70)
print(f"• pass@{K_RUNS}: 1 - C(n-c,k)/C(n,k) — probability of ≥1 success")
print(f"• pass^{K_RUNS}: C(c,k)/C(n,k) — probability ALL k samples succeed")
print()
summary_tok_df

Pass Metrics Summary (first 5 runs per model)
• pass@5: 1 - C(n-c,k)/C(n,k) — probability of ≥1 success
• pass^5: C(c,k)/C(n,k) — probability ALL k samples succeed



,Model,Agent,Mean %,pass@5 %,pass^5 %
0,gpt-5-1-codex-max,GitHub Copilot CLI,48.7,67.3,32.7
1,gpt-5-1-codex-min,GitHub Copilot CLI,45.1,67.3,25.5
2,haiku-4-5,GitHub Copilot CLI,44.7,67.3,23.6
3,haiku-4-5-altool,GitHub Copilot CLI,47.6,76.4,23.6
4,mini-haiku-4-5,mini-bc-agent,37.5,65.5,10.9
5,mini-opus-4-5,mini-bc-agent,60.4,72.7,43.6
6,opus-4-5,GitHub Copilot CLI,59.6,81.8,34.5


In [11]:
# Visualize how mean and pass^k stabilize as runs increase
# Helps answer: "How many runs do we need?"

SELECTED_MODELS = ["mini-opus-4-5", "opus-4-5", "gpt-5-1-codex-max"]
selected_results = {k: v for k, v in all_results.items() if k in SELECTED_MODELS}

fig = make_subplots(rows=2, cols=1, subplot_titles=("Mean Resolution % (±95% CI)", "pass^k %"), vertical_spacing=0.12)

colors = {"mini-opus-4-5": "coral", "opus-4-5": "steelblue", "gpt-5-1-codex-max": "green"}


def bootstrap_metric(pivot_df: pd.DataFrame, run_ids: list, metric_fn, n_bootstrap: int = 2000) -> dict:
    """Bootstrap CI for any metric computed from a pivot table of runs."""
    rng = np.random.default_rng(42)
    n_runs = len(run_ids)
    n_instances = len(pivot_df)

    bootstrap_values = []
    for _ in range(n_bootstrap):
        # Resample runs with replacement
        sampled_runs = rng.choice(run_ids, size=n_runs, replace=True)
        sampled_pivot = pivot_df[sampled_runs]
        bootstrap_values.append(metric_fn(sampled_pivot, n_runs, n_instances))

    bootstrap_values = np.array(bootstrap_values)
    return {
        "ci_low": float(np.percentile(bootstrap_values, 2.5)),
        "ci_high": float(np.percentile(bootstrap_values, 97.5)),
    }


for model_name, model_df in selected_results.items():
    run_ids_sorted = sorted(model_df["run_id"].unique())
    n_runs_total = len(run_ids_sorted)
    n_instances = model_df["instance_id"].nunique()

    # Build pivot for calculations
    pivot = model_df.pivot_table(index="instance_id", columns="run_id", values="resolved", aggfunc=lambda x: x.iloc[0])
    pivot = pivot[run_ids_sorted]

    k_values = list(range(1, n_runs_total + 1))
    mean_pcts, mean_ci_low, mean_ci_high = [], [], []
    pass_hat_k_pcts = []

    for k in k_values:
        first_k_runs = run_ids_sorted[:k]
        pivot_k = pivot[first_k_runs]

        # Mean metric
        def mean_metric(p, n, ni):
            return float(p.mean(axis=1).mean() * 100)

        mean_pct = mean_metric(pivot_k, k, n_instances)
        mean_pcts.append(mean_pct)

        # pass^k metric: C(success,k)/C(n,k) probability
        total = sum(pass_hat_k(k, int(row.sum()), k) for _, row in pivot_k.iterrows())
        pass_hat_k_pcts.append(float(total / n_instances * 100))

        # Bootstrap CI for mean (only meaningful if k >= 2)
        if k >= 2:
            mean_ci = bootstrap_metric(pivot_k, first_k_runs, mean_metric)
            mean_ci_low.append(mean_ci["ci_low"])
            mean_ci_high.append(mean_ci["ci_high"])
        else:
            mean_ci_low.append(mean_pct)
            mean_ci_high.append(mean_pct)

    color = colors.get(model_name, "gray")

    # Mean plot with CI band
    fig.add_trace(
        go.Scatter(
            x=k_values + k_values[::-1], y=mean_ci_high + mean_ci_low[::-1], fill="toself", fillcolor=color, opacity=0.2, line={"width": 0}, showlegend=False, legendgroup=model_name, hoverinfo="skip"
        ),
        row=1,
        col=1,
    )
    fig.add_trace(go.Scatter(x=k_values, y=mean_pcts, mode="lines+markers", name=model_name, line={"color": color}, marker={"size": 6}, legendgroup=model_name), row=1, col=1)

    # pass^k plot (no CI)
    fig.add_trace(go.Scatter(x=k_values, y=pass_hat_k_pcts, mode="lines+markers", name=model_name, line={"color": color}, marker={"size": 6}, legendgroup=model_name, showlegend=False), row=2, col=1)

fig.update_xaxes(title_text="Number of Runs", dtick=2, row=2, col=1)
fig.update_yaxes(title_text="Resolution Rate (%)")

fig.update_layout(height=600, width=800, title_text="How Many Runs Do We Need? Metrics Stabilization", legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "center", "x": 0.5})
fig.show()

print("\n🔍 Interpretation:")
print("   • Mean (top): Expected single-run performance — CI shrinks as runs increase")
print("   • pass^k (bottom): C(success,k)/C(n,k) — probability k samples would all succeed")


🔍 Interpretation:
   • Mean (top): Expected single-run performance — CI shrinks as runs increase
   • pass^k (bottom): C(success,k)/C(n,k) — probability k samples would all succeed


In [12]:
import plotly.graph_objects as go

ci_data = []
for model_name, model_df in all_results.items():
    per_run_means = model_df.groupby("run_id")["resolved"].mean().values * 100  # type: ignore
    if len(per_run_means) >= 2:
        boot = bootstrap_ci(np.array(per_run_means))
        avg_duration = model_df["execution_time"].mean() if "execution_time" in model_df.columns else None
        ci_data.append(
            {
                "Model": model_name,
                "Mean": boot["mean"],
                "CI_low": boot["ci_low"],
                "CI_high": boot["ci_high"],
                "CI_half": boot["ci_half"],
                "Avg Duration (s)": avg_duration,
            }
        )

ci_df = pd.DataFrame(ci_data).sort_values("Mean", ascending=True)

# Dynamic x-axis range: start a bit below the lowest CI, end a bit above highest
x_min = max(0, ci_df["CI_low"].min() - 3)
x_max = ci_df["CI_high"].max() + 3

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ci_df["Mean"],
        y=ci_df["Model"],
        mode="markers",
        marker={"size": 12, "color": "steelblue"},
        error_x={
            "type": "data",
            "symmetric": False,
            "array": ci_df["CI_high"] - ci_df["Mean"],
            "arrayminus": ci_df["Mean"] - ci_df["CI_low"],
            "color": "steelblue",
            "thickness": 2,
            "width": 8,
        },
        name="Mean ± 95% CI",
    )
)

# Add vertical gridlines for easier reading
fig.update_layout(
    title="Model Comparison: Mean Resolution Rate with 95% Bootstrap CI",
    xaxis_title="Resolution Rate (%)",
    yaxis_title="",
    height=max(350, len(ci_df) * 50),
    xaxis={
        "range": [x_min, x_max],
        "dtick": 5,  # gridline every 5%
        "showgrid": True,
        "gridcolor": "lightgray",
    },
    yaxis={"showgrid": False},
    showlegend=False,
    margin={"l": 150},  # more space for model names
)
fig.show()

# Display table with mean ± CI format and average duration
table_df = ci_df.copy()
table_df["Mean ± 95% CI"] = table_df.apply(lambda r: f"{r['Mean']:.1f}% ± {r['CI_half']:.1f}%", axis=1)
table_df["Avg Duration"] = table_df["Avg Duration (s)"].apply(lambda x: f"{x:.0f}s" if pd.notna(x) else "N/A")
table_df = table_df[["Model", "Mean ± 95% CI", "Avg Duration"]].reset_index(drop=True)
table_df

,Model,Mean ± 95% CI,Avg Duration
0,mini-haiku-4-5,36.5% ± 3.6%,193s
1,haiku-4-5,45.1% ± 2.1%,150s
2,gpt-5-1-codex-min,45.8% ± 1.9%,285s
3,haiku-4-5-altool,49.3% ± 2.3%,269s
4,gpt-5-1-codex-max,50.9% ± 2.5%,152s
5,opus-4-5,59.6% ± 1.3%,134s
6,mini-opus-4-5,62.2% ± 1.6%,331s
